# Data Structures

In [2]:
from dataclasses import dataclass
from enum import StrEnum


class EditOperation(StrEnum):
    """Type of edit operation between source and target words."""

    KEEP = "K"
    REPLACE = "R"
    INSERT = "I"
    DELETE = "D"
    MERGE = "M"
    SPLIT = "S"


class AlignmentType(StrEnum):
    """Type of alignment."""

    WORD = "word"
    SENTENCE = "sentence"


@dataclass
class Alignment:
    """Represents an alignment between source and target word spans.

    Stores the alignment between a span of source words and a span of target
    words, along with the edit operation.
    """

    source_start: int
    source_end: int

    target_start: int
    target_end: int

    operation: EditOperation
    label: str | None = None
    alignment_type: AlignmentType = AlignmentType.WORD


@dataclass
class BackPointer:
    """Represents a back pointer in the DP table.

    Stores the operation and previous indices for backtracking through the
    dynamic programming table.
    """

    operation: EditOperation

    prev_i: int
    prev_j: int

# 1. Alignement

In [ ]:
class Aligner:
    """Aligns two lists of words using dynamic programming."""

    INSERT_DELETE_COST = 1
    REPLACE_COST = 2
    MERGE_COST = 1
    SPLIT_COST = 1

    def align_words(self, source: str, target: str) -> list[Alignment]:
        """Align two strings at word level."""
        source = str.split(source, " ")
        target = str.split(target, " ")
        _, parent = self._build_dp(source, target)
        return self._backtrack(source, target, parent)

trying some distance and similarity metrics to see how they perform on the task of aligning two sentences and extracting the edits between them.

In [4]:
from difflib import SequenceMatcher


def similarity(a: str, b: str) -> float:
    """Calculate the similarity ratio between two strings using SequenceMatcher."""
    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


def common_prefix_ratio(a, b):
    """Calculate the ratio of common prefix length to max length."""
    count = 0

    for x, y in zip(a, b, strict=False):
        if x != y:
            break

        count += 1

    return count / max(len(a), len(b))

In [29]:
source = "I amlove pythin"
target = "I am lo ve python i"

aligner = Aligner()

alignments = aligner.align_words(source, target)
print("word alignments:")
for alignment in alignments:
    print(alignment)

print("char alignments:")
alignments = aligner.align_characters(source, target)
for alignment in alignments:
    print(alignment)

word alignments:
Alignment(source_start=0, source_end=0, target_start=0, target_end=0, operation=<EditOperation.KEEP: 'K'>, label=None, alignment_type=<AlignmentType.WORD: 'word'>)
Alignment(source_start=1, source_end=1, target_start=1, target_end=3, operation=<EditOperation.SPLIT: 'S'>, label=None, alignment_type=<AlignmentType.WORD: 'word'>)
Alignment(source_start=2, source_end=2, target_start=4, target_end=4, operation=<EditOperation.REPLACE: 'R'>, label='python', alignment_type=<AlignmentType.WORD: 'word'>)
Alignment(source_start=3, source_end=2, target_start=5, target_end=5, operation=<EditOperation.INSERT: 'I'>, label='i', alignment_type=<AlignmentType.WORD: 'word'>)
char alignments:
Alignment(source_start=0, source_end=0, target_start=0, target_end=0, operation=<EditOperation.KEEP: 'K'>, label=None, alignment_type=<AlignmentType.WORD: 'word'>)
Alignment(source_start=1, source_end=1, target_start=1, target_end=1, operation=<EditOperation.KEEP: 'K'>, label=None, alignment_type=<Al

# Edit Compression

In [6]:
class Extractor:
    """Extracts edit tags from word alignments."""

    def extract_tags(self, alignment: list[Alignment]) -> list[str]:
        """Compresses a word alignment into an edit tag string."""
        tags = list[str]()
        for a in alignment:
            if a.operation == EditOperation.KEEP:
                tags.append("k")
            elif a.operation == EditOperation.REPLACE:
                label = a.label
                if label is None:
                    raise ValueError("Label cannot be None for REPLACE operation")
                tags.append(f"r_[{label}]")

            elif a.operation == EditOperation.INSERT:
                label = a.label
                if label is None:
                    raise ValueError("Label cannot be None for INSERT operation")
                tags.append(f"i_[{label}]")

            elif a.operation == EditOperation.DELETE:
                tags.append("d")
            elif a.operation == EditOperation.MERGE:
                tags.append("m")
            elif a.operation == EditOperation.SPLIT:
                tags.append("s")
            else:
                raise ValueError(f"Unknown operation: {a.operation}")
        return tags

In [7]:
class Compressor:
    """Compresses tag strings by merging consecutive identical tags."""

    def compress_tags(self, tags: list[str]) -> str:
        """Compresses a tag string by merging consecutive identical tags."""
        if not tags:
            return ""

        compressed = []
        count = 1
        prev_tag = tags[0]

        for tag in tags[1:]:
            if tag == prev_tag:
                count += 1
            else:
                compressed.append(f"{prev_tag}*" if count > 1 else prev_tag)
                prev_tag = tag
                count = 1

        compressed.append(f"{prev_tag}*" if count > 1 else prev_tag)
        return "".join(compressed)

In [8]:
extractor = Extractor()
compressor = Compressor()
tags = extractor.extract_tags(alignments)
print(tags)
print(compressor.compress_tags(tags))
print(compressor.compress_tags(["k", "k", "d", "k", "r_[love]", "r_[love]", "k", "k"]))

['k', 'd', 'm', 's']
kdms
k*dkr_[love]*k*


# Subword Projection

In [9]:
class SubwordProjection:
    """Projects word-level alignments to subword level."""

    def tokenize(self, tokens: list[str]) -> list[str]:
        """Tokenizes a word into subwords."""
        # TODO: use preprocessing tokens when ready
        return ["سُم", "يه"]
        pass

    def compute_spans(self, subwords: list[str]) -> list[tuple[int, int]]:
        """Computes the character spans of each subword within the word."""
        # TODO: ARABert returns a ## for subwords, that needs to be handled (if i used
        # ARABert version of tokenization)
        # in span computation in case we used the ARABert version of
        # tokenization, otherwise, nope?

        current_ind = 0
        spans = []
        for subword in subwords:
            clean_subword = subword.replace("##", "")
            start = current_ind
            end = start + len(clean_subword) - 1
            spans.append((start, end))
            current_ind = end + 1
        return spans

    def find_corresponding_subword_edit(
        self, char_ind: int, spans: list[tuple[int, int]]
    ) -> int:
        """Finds the index of the subword that contains the given character index."""
        for i, (start, end) in enumerate(spans):
            if start <= char_ind <= end:
                return i
        raise ValueError(f"Character index {char_ind} out of bounds")

    def project(self, word: str, edits: list[Alignment]) -> list[list[Alignment]]:
        """Projects word-level edits to subword level."""
        subwords = self.tokenize(word)
        spans = self.compute_spans(subwords)
        projection = [[] for _ in subwords]

        for edit in edits:
            subword_ind = self.find_corresponding_subword_edit(edit.source_start, spans)
            projection[subword_ind].append(edit)
        return projection

    def compress_projection(
        self,
        projections: list[list[Alignment]],
        extractor: Extractor,
        compressor: Compressor,
    ) -> list[str]:
        """Compresses tags per subword for each word."""
        compressed_tags = []
        for projection in projections:
            tags = extractor.extract_tags(projection)
            compressed_tags.append(compressor.compress_tags(tags))
        return compressed_tags

In [10]:
sub_proj = SubwordProjection()
alignments = aligner.align_characters("سميه", "سُمية")
projection = sub_proj.project("سميه", alignments)
print(sub_proj.compress_projection(projection, extractor, compressor))

['ki_[ُ]k*', 'r_[ة]']
